# Risk-averse mutation analysis — aggregate tables and figures

This notebook reads the output folders produced by `simulation_mutation_risk_averse.ipynb`, assembles all risk settings into one analysis table, pairs each mutation scenario against its same-risk baseline, and exports journal-ready summary tables and figures.

Expected result structure:

```text
Code_new/
  simulation_mutation_averse/
    Output files (Risk Averse, Mutation, Verified)/
      Risk_Averse_Dynamic_Output_Index.csv        # preferred, optional
      01_seller_low__lambdaS_0p253__lambdaB_0p000/
        Simulation_Plot_Data_All_Matches.csv
        Simulation_Run_Status.csv
        Hourly_FE_Index.csv
      ...
```

The notebook can also scan the risk-output subfolders if the root index file is missing or incomplete.

In [ ]:
# ============================================================
# SECTION 1. USER CONTROLS
# ============================================================
from pathlib import Path

# -----------------------------
# Main path controls
# -----------------------------
# Leave these as None when this notebook is placed in either:
#   Code_new/                         or
#   Code_new/simulation_mutation_averse/
code_root_override = None
risk_averse_mutation_dir_override = None
risk_output_base_dir_override = None

risk_averse_mutation_folder_name = "simulation_mutation_averse"
risk_output_base_folder_name = "Output files (Risk Averse, Mutation, Verified)"
risk_output_index_filename = "Risk_Averse_Dynamic_Output_Index.csv"

analysis_output_folder_name = "Risk_Averse_Mutation_Analysis"
table_output_subdir = "tables"
figure_output_subdir = "figures"

# Optional risk-neutral benchmark. The analysis runs without it.
include_risk_neutral_reference = True
risk_neutral_plot_data_csv_override = None
risk_neutral_folder_name = "simulation_mutation"
risk_neutral_output_folder_name = "Output files (Risk Neutral, Mutation, Verified)"
risk_neutral_plot_data_filename = "Simulation_Plot_Data_All_Matches.csv"

# -----------------------------
# Analysis filters
# -----------------------------
# Leave as None to keep all available rows.
filter_risk_labels = None                  # e.g. ["seller_low", "buyer_high"]
filter_risk_modes = None                   # e.g. ["Seller risk-averse", "Buyer risk-averse", "Joint risk-averse"]
filter_lambda_s_values = None              # e.g. [0.253, 0.524]
filter_lambda_b_values = None
filter_match_ids = None                    # e.g. [1, 2, 3]
filter_mutation_families = None            # e.g. ["shape", "basis", "load_price", "cannibalization"]
filter_target_physical_shifts = None       # e.g. [-0.10, -0.30, -0.50, 0.10, 0.30, 0.50]
filter_scenario_names = None

# If True, retain only summary rows with a success row in Simulation_Run_Status.csv.
# This is useful when a simulation run partially failed because sample files were missing.
require_success_in_run_status = False

# Baseline rows are needed internally for paired deltas. This controls whether
# baseline rows are also written to the standardized all-row export.
include_baseline_rows_in_standardized_export = True

# -----------------------------
# Column-selection controls
# -----------------------------
# The first existing numeric column in each list is used.
seller_metric_column_candidates = [
    "seller_utility", "seller_risk_adjusted_objective", "seller_objective",
    "seller_objective_value", "seller_mean_exposure", "seller_risk_adjusted_exposure",
]
buyer_metric_column_candidates = [
    "buyer_utility", "buyer_risk_adjusted_objective", "buyer_objective",
    "buyer_objective_value", "buyer_mean_exposure", "buyer_risk_adjusted_exposure",
]
buyer_slack_column_candidates = [
    "buyer_participation_slack", "participation_slack", "buyer_slack", "buyer_feasibility_slack",
]
strike_price_column_candidates = [
    "strike_price_mwh", "selected_strike_price_mwh", "strike_price", "price_mwh", "contract_price_mwh", "pi",
]
volume_column_candidates = [
    "volume_mw", "selected_volume_mw", "fixed_volume_mw", "contract_volume_mw", "q_mw", "q",
]
generation_mean_column_candidates = [
    "generation_simulated_mean", "generation_mean_ref", "generation_original_mean", "generation_mean",
    "seller_volume_mean_used", "mean_generation_mw", "mean_g_mw", "g_mean",
]
demand_mean_column_candidates = [
    "demand_simulated_mean", "demand_mean_ref", "demand_original_mean", "demand_mean",
    "buyer_volume_mean_used", "mean_demand_mw", "mean_d_mw", "d_mean",
]

# Metrics exported in distribution and response tables. Missing metrics are skipped.
metrics_to_summarize = [
    "delta_strike_price_mwh",
    "delta_contract_volume_mw",
    "delta_fixed_volume_mw",
    "delta_delivered_volume_proxy_mw",
    "delta_seller_metric",
    "delta_buyer_metric",
    "delta_total_metric",
    "delta_buyer_participation_slack",
]

# -----------------------------
# Ordering and labels
# -----------------------------
profile_order = ["Fix", "AsC", "AsG", "No Contract", "Unknown"]
family_order = [
    "Profile-shape deterioration",
    "Basis deterioration",
    "Buyer load-price intensification",
    "Seller-side cannibalization",
    "Baseline",
    "Unknown",
]
risk_mode_order = ["Seller risk-averse", "Buyer risk-averse", "Joint risk-averse", "Risk neutral", "Other"]

family_display_labels = {
    "shape": "Profile-shape deterioration",
    "basis": "Basis deterioration",
    "load_price": "Buyer load-price intensification",
    "cannibalization": "Seller-side cannibalization",
    "baseline": "Baseline",
}

# Display-only titles for the metric-interval figure columns.
family_panel_title_labels = {
    "Profile-shape deterioration": "Generation–load mismatch",
    "Basis deterioration": "Seller–buyer nodal price decoupling",
    "Buyer load-price intensification": "Buyer load–price intensification",
    "Seller-side cannibalization": "Seller generation–price cannibalization",
}

metric_display_labels = {
    "delta_strike_price_mwh": "Δ Strike price ($/MWh)",
    "delta_contract_volume_mw": "Δ Contract volume (MW)",
    "delta_fixed_volume_mw": "Δ Fixed volume (MW; Fix→Fix)",
    "delta_delivered_volume_proxy_mw": "Δ Mean delivered volume (MW)",
    "delta_seller_metric": "Δ Seller exposure index",
    "delta_buyer_metric": "Δ Buyer exposure index",
    "delta_total_metric": "Δ Seller + Buyer metric",
    "delta_buyer_participation_slack": "Δ Buyer participation slack",
}

In [ ]:
# ============================================================
# SECTION 2. FIGURE CONTROLS
# ============================================================
make_figures = True
save_png = True
save_pdf = False
save_svg = False
display_figures_in_notebook = True
show_preview_tables = True
figure_dpi = 600

# Figure switches.
make_run_status_figure = True
make_baseline_profile_by_risk_figure = True
make_profile_share_heatmaps = True
make_switch_rate_heatmap = True
make_metric_delta_interval_figures = True
make_risk_response_figures = True
make_risk_neutral_difference_figures = True

# Profile heatmaps: each listed profile gets one heatmap.
profile_share_heatmap_profiles = ["Fix", "AsC", "AsG", "No Contract"]
profile_share_heatmap_max_columns = 18

# Switch heatmap.
switch_heatmap_value = "profile_switch_rate"  # "profile_switch_rate" or "contract_type_switch_rate"

# Metric interval figures.
metric_interval_figure_metrics = [
    "delta_strike_price_mwh",
    "delta_fixed_volume_mw",
    "delta_delivered_volume_proxy_mw",
    "delta_seller_metric",
    "delta_buyer_metric",
    "delta_buyer_participation_slack",
]
metric_interval_plot_stat = "median"          # "mean" or "median"
metric_interval_error_band = "iqr"            # "iqr", "p10_p90", or None
metric_interval_risk_modes = None              # None = all available modes; or e.g. ["Joint risk-averse"]
metric_interval_panel_width = 3.0              # compact inches per family panel
metric_interval_joint_panel_width = 4.375      # 17.5 inches total for four joint-risk panels
metric_interval_figure_height = 5.0
metric_interval_axis_label_fontsize = 16
metric_interval_tick_label_fontsize = 14
metric_interval_panel_title_fontsize = 17
metric_interval_legend_fontsize = 14
metric_interval_wspace = 0.18

# Risk-response figures use the most severe available target shift in each mutation family.
risk_response_metrics = ["profile_switch_rate", "ppa_formation_rate", "delta_seller_metric_median", "delta_buyer_metric_median"]
risk_response_risk_modes = ["Seller risk-averse", "Buyer risk-averse", "Joint risk-averse"]

# Generic style controls.
default_fig_width = 12.0
default_row_height = 3.6
minimum_fig_height = 4.2
axis_label_fontsize = 12
tick_label_fontsize = 12
panel_title_fontsize = 12
legend_fontsize = 12
x_tick_rotation = 35
heatmap_annotation_fontsize = 10
heatmap_annotation_format = ".0%"              # values are usually rates/shares

# Distribution clipping for table summaries only. Figures use all finite observations.
summary_quantiles = [0.10, 0.25, 0.50, 0.75, 0.90]

In [ ]:
# ============================================================
# SECTION 3. IMPORTS, PATH RESOLUTION, AND OUTPUT FOLDERS
# ============================================================
from __future__ import annotations

import json
import math
import re
import textwrap
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

try:
    from IPython.display import display
except Exception:  # pragma: no cover
    display = print


NOTEBOOK_CWD = Path.cwd().resolve()


def _as_path_or_none(value) -> Optional[Path]:
    if value is None:
        return None
    text = str(value).strip()
    if text == "" or text.lower() in {"none", "nan", "<na>"}:
        return None
    return Path(text).expanduser().resolve()


def _dedupe_paths(paths: Iterable[Path]) -> list[Path]:
    out = []
    seen = set()
    for p in paths:
        try:
            rp = p.expanduser().resolve()
        except Exception:
            continue
        if str(rp) not in seen:
            out.append(rp)
            seen.add(str(rp))
    return out


def resolve_code_root() -> Path:
    override = _as_path_or_none(code_root_override)
    if override is not None:
        return override
    cwd = NOTEBOOK_CWD
    candidates = [cwd, cwd.parent, cwd / "Code_Submission"]
    if cwd.name == risk_averse_mutation_folder_name:
        candidates.insert(0, cwd.parent)
    if cwd.name == "Code_Submission":
        candidates.insert(0, cwd)
    candidates.extend(parent / "Code_Submission" for parent in cwd.parents)
    for c in _dedupe_paths(candidates):
        if (
            (c / "simulation_mutation").is_dir()
            and (c / risk_averse_mutation_folder_name).is_dir()
            and (c / "Input data and files").is_dir()
        ):
            return c
    raise FileNotFoundError(
        "Could not locate the Code_Submission root from the notebook working directory: "
        f"{NOTEBOOK_CWD}"
    )


def resolve_risk_averse_dir(code_root: Path) -> Path:
    override = _as_path_or_none(risk_averse_mutation_dir_override)
    if override is not None:
        return override
    candidates = [
        NOTEBOOK_CWD,
        NOTEBOOK_CWD / risk_averse_mutation_folder_name,
        NOTEBOOK_CWD.parent / risk_averse_mutation_folder_name,
        code_root / risk_averse_mutation_folder_name,
    ]
    if NOTEBOOK_CWD.name == risk_averse_mutation_folder_name:
        candidates.insert(0, NOTEBOOK_CWD)
    for c in _dedupe_paths(candidates):
        if (c / risk_output_base_folder_name).exists():
            return c
    # Return the conventional location even if it does not exist yet; downstream error will be explicit.
    return (code_root / risk_averse_mutation_folder_name).resolve()


CODE_ROOT = resolve_code_root()
RISK_AVERSE_DIR = resolve_risk_averse_dir(CODE_ROOT)

risk_base_override = _as_path_or_none(risk_output_base_dir_override)
RISK_OUTPUT_BASE = risk_base_override if risk_base_override is not None else (RISK_AVERSE_DIR / risk_output_base_folder_name).resolve()
ANALYSIS_OUTPUT_DIR = RISK_OUTPUT_BASE / analysis_output_folder_name
TABLE_DIR = ANALYSIS_OUTPUT_DIR / table_output_subdir
FIGURE_DIR = ANALYSIS_OUTPUT_DIR / figure_output_subdir
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Resolved folders:")
print(f"  Notebook cwd             : {NOTEBOOK_CWD}")
print(f"  Code root                : {CODE_ROOT}")
print(f"  Risk-averse mutation dir : {RISK_AVERSE_DIR}")
print(f"  Risk output base         : {RISK_OUTPUT_BASE}")
print(f"  Table output dir         : {TABLE_DIR}")
print(f"  Figure output dir        : {FIGURE_DIR}")

In [ ]:
# ============================================================
# SECTION 4. INPUT DISCOVERY AND LOADING
# ============================================================

def _read_csv_if_exists(path: Path, **kwargs) -> pd.DataFrame:
    if path is None:
        return pd.DataFrame()
    p = Path(path)
    if not p.exists() or (p.is_file() and p.stat().st_size == 0):
        return pd.DataFrame()
    try:
        return pd.read_csv(p, **kwargs)
    except pd.errors.EmptyDataError:
        return pd.DataFrame()


def _path_from_record(value, output_base: Path) -> Optional[Path]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return None
    text = str(value).strip()
    if not text or text.lower() in {"nan", "none", "<na>"}:
        return None
    p = Path(text).expanduser()
    candidates = []
    if p.is_absolute():
        candidates.append(p)
        candidates.append(output_base / p.name)
        if len(p.parts) >= 2:
            candidates.append(output_base / p.parts[-2] / p.name)
    else:
        candidates += [output_base / p, RISK_AVERSE_DIR / p, CODE_ROOT / p, NOTEBOOK_CWD / p]
    for c in _dedupe_paths(candidates):
        if c.exists():
            return c
    return p.resolve() if p.is_absolute() else (output_base / p).resolve()


def _parse_risk_from_folder(folder: Path) -> dict:
    name = folder.name
    out = {"risk_folder": name, "risk_setting_order": np.nan, "risk_label": name, "lambda_s": np.nan, "lambda_b": np.nan}
    m_order = re.match(r"^(\d+)_", name)
    if m_order:
        out["risk_setting_order"] = int(m_order.group(1))
    m = re.search(r"^(?:\d+_)?(?P<label>.*?)__lambdaS_(?P<ls>[^_]+)__lambdaB_(?P<lb>[^_]+)$", name)
    if m:
        out["risk_label"] = m.group("label")
        out["lambda_s"] = float(m.group("ls").replace("m", "-").replace("p", "."))
        out["lambda_b"] = float(m.group("lb").replace("m", "-").replace("p", "."))
    return out


In [ ]:
# ============================================================
# SECTION 4.5. SIMULATION OUTPUT DISCOVERY AND ASSEMBLY
# ============================================================

def discover_risk_outputs() -> pd.DataFrame:
    records = []
    index_path = RISK_OUTPUT_BASE / risk_output_index_filename

    if index_path.exists():
        index_df = _read_csv_if_exists(index_path)
        for _, row in index_df.iterrows():
            output_root = _path_from_record(row.get("output_root", row.get("output_dir", None)), RISK_OUTPUT_BASE)
            summary_path = _path_from_record(row.get("summary_path", None), RISK_OUTPUT_BASE)
            if (summary_path is None or not summary_path.exists()) and output_root is not None:
                candidate = output_root / "Simulation_Plot_Data_All_Matches.csv"
                if candidate.exists():
                    summary_path = candidate
            if output_root is None and summary_path is not None:
                output_root = summary_path.parent
            if summary_path is None or not summary_path.exists():
                continue
            rec = {
                "risk_setting_order": row.get("risk_order", row.get("risk_setting_order", np.nan)),
                "risk_label": row.get("risk_label", np.nan),
                "lambda_s": row.get("lambda_s", np.nan),
                "lambda_b": row.get("lambda_b", np.nan),
                "output_root": output_root,
                "summary_path": summary_path,
                "run_status_path": output_root / "Simulation_Run_Status.csv",
                "case_summary_path": output_root / "Mutation_Case_Summary.csv",
                "switch_long_path": output_root / "Mutation_Profile_Switch_Long.csv",
                "hourly_fe_index_path": output_root / "Hourly_FE_Index.csv",
                "discovery_source": "index",
            }
            if output_root is not None:
                parsed = _parse_risk_from_folder(output_root)
                for key, val in parsed.items():
                    if pd.isna(rec.get(key, np.nan)) or rec.get(key, "") in {"", "nan"}:
                        rec[key] = val
            records.append(rec)

    if RISK_OUTPUT_BASE.exists():
        for folder in sorted([p for p in RISK_OUTPUT_BASE.iterdir() if p.is_dir()]):
            summary_path = folder / "Simulation_Plot_Data_All_Matches.csv"
            if not summary_path.exists():
                continue
            if any(Path(rec["summary_path"]) == summary_path for rec in records if rec.get("summary_path") is not None):
                continue
            rec = _parse_risk_from_folder(folder)
            rec.update({
                "output_root": folder,
                "summary_path": summary_path,
                "run_status_path": folder / "Simulation_Run_Status.csv",
                "case_summary_path": folder / "Mutation_Case_Summary.csv",
                "switch_long_path": folder / "Mutation_Profile_Switch_Long.csv",
                "hourly_fe_index_path": folder / "Hourly_FE_Index.csv",
                "discovery_source": "folder_scan",
            })
            records.append(rec)

    if not records:
        raise FileNotFoundError(
            f"No risk-averse simulation outputs were found under {RISK_OUTPUT_BASE}. "
            "Expected subfolders containing Simulation_Plot_Data_All_Matches.csv."
        )

    out = pd.DataFrame(records)
    for col in ["risk_setting_order", "lambda_s", "lambda_b"]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    out = out.sort_values(["risk_setting_order", "risk_label"], na_position="last").reset_index(drop=True)
    return out


def _load_table_from_inventory(inventory: pd.DataFrame, field: str, table_name: str) -> pd.DataFrame:
    frames = []
    for row in inventory.itertuples(index=False):
        path = getattr(row, field, None)
        if path is None:
            continue
        table = _read_csv_if_exists(path)
        if table.empty:
            print(f"Skipped empty or missing {table_name} for {getattr(row, 'risk_label', 'unknown')} at {path}")
            continue
        for col in ["risk_label", "lambda_s", "lambda_b", "risk_setting_order"]:
            if col not in table.columns and hasattr(row, col):
                table[col] = getattr(row, col)
        frames.append(table)
    return pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()


risk_inventory = discover_risk_outputs()
summary_raw = _load_table_from_inventory(risk_inventory, "summary_path", "summary table")
run_status_raw = _load_table_from_inventory(risk_inventory, "run_status_path", "run-status table")
case_summary_raw = _load_table_from_inventory(risk_inventory, "case_summary_path", "case-summary table")
switch_long_raw = _load_table_from_inventory(risk_inventory, "switch_long_path", "switch-long table")
hourly_fe_index_raw = _load_table_from_inventory(risk_inventory, "hourly_fe_index_path", "hourly FE index table")

print("Discovered risk outputs:", len(risk_inventory))
print("Summary rows:", len(summary_raw))
print("Run-status rows:", len(run_status_raw))
print("Case-summary rows:", len(case_summary_raw))
print("Switch-long rows:", len(switch_long_raw))
print("Hourly FE index rows:", len(hourly_fe_index_raw))


In [ ]:
# ============================================================
# SECTION 5. COLUMN NORMALIZATION AND FILTER HELPERS
# ============================================================

def _clean_text_series(s: pd.Series) -> pd.Series:
    return s.astype("string").fillna("").str.strip()


def _num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")


def _first_existing_column(df: pd.DataFrame, candidates: list[str]) -> Optional[str]:
    for col in candidates:
        if col in df.columns:
            return col
    return None


def _first_numeric_series(df: pd.DataFrame, candidates: list[str], default=np.nan) -> tuple[pd.Series, Optional[str]]:
    col = _first_existing_column(df, candidates)
    if col is None:
        return pd.Series(default, index=df.index, dtype="float64"), None
    return _num(df[col]), col


def normalize_profile_value(value) -> str:
    text = str(value).strip()
    low = text.lower().replace("_", " ").replace("-", " ")
    if low in {"fix", "fixed", "physical fix", "virtual fix"}:
        return "Fix"
    if low in {"asc", "as contracted", "as contracted generation", "as contracted load"}:
        return "AsC"
    if low in {"asg", "as generated", "as generated generation"}:
        return "AsG"
    if low in {"no contract", "none", "nan", "", "n/a", "na"}:
        return "No Contract"
    return text


def normalize_ppa_type(value) -> str:
    text = str(value).strip()
    low = text.lower().replace("_", " ").replace("-", " ")
    if low in {"physical", "phys"}:
        return "Physical"
    if low in {"virtual", "financial"}:
        return "Virtual"
    if low in {"no contract", "none", "nan", "", "n/a", "na"}:
        return "No Contract"
    return text


def normalize_family_value(value) -> str:
    text = str(value).strip()
    low = text.lower().replace("-", "_").replace(" ", "_")
    if low in family_display_labels:
        return family_display_labels[low]
    if "shape" in low:
        return family_display_labels["shape"]
    if "basis" in low:
        return family_display_labels["basis"]
    if "load" in low or "buyer" in low:
        return family_display_labels["load_price"]
    if "cannibal" in low or "seller" in low:
        return family_display_labels["cannibalization"]
    if "baseline" in low or low in {"", "nan", "none"}:
        return "Baseline"
    return text


def risk_mode(lambda_s, lambda_b) -> str:
    ls = float(lambda_s) if pd.notna(lambda_s) else 0.0
    lb = float(lambda_b) if pd.notna(lambda_b) else 0.0
    eps = 1e-10
    if abs(ls) <= eps and abs(lb) <= eps:
        return "Risk neutral"
    if abs(ls) > eps and abs(lb) <= eps:
        return "Seller risk-averse"
    if abs(ls) <= eps and abs(lb) > eps:
        return "Buyer risk-averse"
    if abs(ls) > eps and abs(lb) > eps:
        return "Joint risk-averse"
    return "Other"


def risk_label_display(label, lambda_s, lambda_b) -> str:
    label_text = str(label).strip() if str(label).strip() else "risk"
    lambda_s_display = float(lambda_s)
    lambda_b_display = float(lambda_b)
    canonical_risk_levels = {"low": 0.253, "medium": 0.524, "high": 0.842}
    label_lower = label_text.lower()
    for level, canonical_value in canonical_risk_levels.items():
        if label_lower == level or label_lower.endswith(f"_{level}"):
            if abs(lambda_s_display) > 1e-10:
                lambda_s_display = canonical_value
            if abs(lambda_b_display) > 1e-10:
                lambda_b_display = canonical_value
            break
    return f"{label_text} (λs={lambda_s_display:.3f}, λb={lambda_b_display:.3f})"


def risk_legend_label(label, lambda_s, lambda_b) -> str:
    """Format a risk setting for plot legends without changing table labels."""
    raw_label_text = str(label).strip() if str(label).strip() else "risk"
    label_text = raw_label_text.replace("_", " ")
    label_text = label_text[:1].upper() + label_text[1:]
    if not label_text.lower().endswith("risk aversion"):
        label_text = f"{label_text} risk aversion"
    lambda_s_display = float(lambda_s) if pd.notna(lambda_s) else 0.0
    lambda_b_display = float(lambda_b) if pd.notna(lambda_b) else 0.0
    canonical_risk_levels = {"low": 0.253, "medium": 0.524, "high": 0.842}
    label_lower = raw_label_text.lower()
    for level, canonical_value in canonical_risk_levels.items():
        if label_lower == level or label_lower.endswith(f"_{level}"):
            if abs(lambda_s_display) > 1e-10:
                lambda_s_display = canonical_value
            if abs(lambda_b_display) > 1e-10:
                lambda_b_display = canonical_value
            break
    seller_active = abs(lambda_s_display) > 1e-10
    buyer_active = abs(lambda_b_display) > 1e-10
    if seller_active and not buyer_active:
        return rf"{label_text} ($\lambda^{{S}}={lambda_s_display:.3f}$)"
    if buyer_active and not seller_active:
        return rf"{label_text} ($\lambda^{{B}}={lambda_b_display:.3f}$)"
    if seller_active and buyer_active and np.isclose(lambda_s_display, lambda_b_display):
        return rf"{label_text} ($\lambda^{{S}}=\lambda^{{B}}={lambda_s_display:.3f}$)"
    return rf"{label_text} ($\lambda^{{S}}={lambda_s_display:.3f}, \lambda^{{B}}={lambda_b_display:.3f}$)"


def infer_scenario_type(df: pd.DataFrame) -> pd.Series:
    if "scenario_type" in df.columns:
        raw = _clean_text_series(df["scenario_type"]).str.lower()
    else:
        raw = pd.Series("", index=df.index, dtype="string")
    scenario = _clean_text_series(df.get("scenario_name", pd.Series("", index=df.index))).str.lower()
    return np.where(raw.eq("baseline") | scenario.str.contains("baseline|no_mutation|no mutation", regex=True), "baseline", "mutation")


def infer_target_shift(df: pd.DataFrame) -> pd.Series:
    for col in ["target_physical_shift", "target_delta", "calibrated_latent_delta", "mutation_level_abs"]:
        if col in df.columns:
            s = _num(df[col])
            if s.notna().any():
                if col == "mutation_level_abs":
                    fam = df.get("mutation_family", pd.Series("", index=df.index)).astype(str).str.lower()
                    sign = np.where(fam.str.contains("load"), 1.0, -1.0)
                    return s.abs() * sign
                return s
    # Last-resort parse from scenario name, e.g. m0p30 or p0p50.
    text = df.get("scenario_name", pd.Series("", index=df.index)).astype(str)
    vals = []
    for item in text:
        m = re.search(r"target[^_]*_shift_([mp])([0-9]+)p([0-9]+)", item)
        if not m:
            vals.append(np.nan)
            continue
        sign = -1.0 if m.group(1) == "m" else 1.0
        vals.append(sign * float(f"{m.group(2)}.{m.group(3)}"))
    return pd.Series(vals, index=df.index, dtype="float64")


def target_shift_label(value) -> str:
    if pd.isna(value):
        return "NA"
    value = float(value)
    if abs(value) < 1e-12:
        return "0"
    return f"{value:+.2f}"


def _as_set(values):
    if values is None:
        return None
    if isinstance(values, (str, int, float)):
        return {values}
    return set(values)


def _numeric_filter(series: pd.Series, allowed, atol: float = 1e-9) -> pd.Series:
    if allowed is None:
        return pd.Series(True, index=series.index)
    allowed_vals = [float(x) for x in (allowed if isinstance(allowed, (list, tuple, set)) else [allowed])]
    s = _num(series)
    keep = pd.Series(False, index=series.index)
    for val in allowed_vals:
        keep = keep | np.isclose(s, val, atol=atol, equal_nan=False)
    return keep


def delivered_volume_proxy(profile: pd.Series, volume: pd.Series, generation_mean: pd.Series, demand_mean: pd.Series) -> pd.Series:
    profile_norm = profile.map(normalize_profile_value)
    vol = _num(volume)
    gen = _num(generation_mean)
    dem = _num(demand_mean)
    out = pd.Series(np.nan, index=profile.index, dtype="float64")
    out.loc[profile_norm.eq("Fix")] = vol.loc[profile_norm.eq("Fix")]
    out.loc[profile_norm.eq("AsG")] = gen.loc[profile_norm.eq("AsG")]
    out.loc[profile_norm.eq("AsC")] = dem.loc[profile_norm.eq("AsC")]
    out.loc[profile_norm.eq("No Contract")] = 0.0
    return out


def standardize_summary(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]
    sources = {}

    for col in ["match_id", "lambda_s", "lambda_b", "risk_setting_order"]:
        if col in out.columns:
            out[col] = _num(out[col])
    if "match_id" in out.columns:
        out = out.loc[out["match_id"].notna()].copy()
        out["match_id"] = out["match_id"].astype(int)

    if "risk_label" not in out.columns:
        out["risk_label"] = "risk_unknown"
    out["risk_label"] = _clean_text_series(out["risk_label"]).replace("", "risk_unknown")
    out["lambda_s"] = _num(out.get("lambda_s", pd.Series(0.0, index=out.index))).fillna(0.0)
    out["lambda_b"] = _num(out.get("lambda_b", pd.Series(0.0, index=out.index))).fillna(0.0)
    out["risk_mode"] = [risk_mode(s, b) for s, b in zip(out["lambda_s"], out["lambda_b"])]
    out["risk_intensity"] = np.maximum(out["lambda_s"].abs(), out["lambda_b"].abs())
    out["risk_setting_key"] = out["risk_label"].astype(str) + "__lambdaS_" + out["lambda_s"].round(6).astype(str) + "__lambdaB_" + out["lambda_b"].round(6).astype(str)
    out["risk_label_display"] = [risk_label_display(l, s, b) for l, s, b in zip(out["risk_label"], out["lambda_s"], out["lambda_b"])]

    if "scenario_name" not in out.columns:
        out["scenario_name"] = "scenario_unknown"
    out["scenario_type"] = infer_scenario_type(out)
    out["target_shift_signed"] = infer_target_shift(out)
    out["target_shift_abs"] = out["target_shift_signed"].abs()
    out["target_shift_label"] = out["target_shift_signed"].map(target_shift_label)

    if "mutation_family" not in out.columns:
        out["mutation_family"] = np.where(out["scenario_type"].eq("baseline"), "baseline", "unknown")
    out["mutation_family_normalized"] = out.get("mutation_family_label", out["mutation_family"]).map(normalize_family_value)
    # If labels are missing/garbled, use mutation_family as fallback.
    fallback = out["mutation_family"].map(normalize_family_value)
    out.loc[out["mutation_family_normalized"].isin(["", "Unknown"]), "mutation_family_normalized"] = fallback
    out.loc[out["scenario_type"].eq("baseline"), "mutation_family_normalized"] = "Baseline"

    ppa_raw = out.get("ppa_type", pd.Series("", index=out.index))
    profile_raw = out.get("profile_type", pd.Series("", index=out.index))
    out["ppa_type_normalized"] = ppa_raw.map(normalize_ppa_type)
    out["profile_type_normalized"] = profile_raw.map(normalize_profile_value)
    out["selected_profile_for_switch"] = np.where(
        out["ppa_type_normalized"].eq("No Contract"),
        "No Contract",
        out["profile_type_normalized"],
    )
    out["active_contract"] = ~out["ppa_type_normalized"].eq("No Contract")

    strike, sources["strike_price_column"] = _first_numeric_series(out, strike_price_column_candidates)
    volume, sources["volume_column"] = _first_numeric_series(out, volume_column_candidates)
    generation_mean, sources["generation_mean_column"] = _first_numeric_series(out, generation_mean_column_candidates)
    demand_mean, sources["demand_mean_column"] = _first_numeric_series(out, demand_mean_column_candidates)
    seller_metric, sources["seller_metric_column"] = _first_numeric_series(out, seller_metric_column_candidates)
    buyer_metric, sources["buyer_metric_column"] = _first_numeric_series(out, buyer_metric_column_candidates)
    buyer_slack, sources["buyer_slack_column"] = _first_numeric_series(out, buyer_slack_column_candidates)

    out["selected_strike_price_mwh"] = strike
    out["selected_volume_mw"] = volume
    out["generation_mean_for_proxy"] = generation_mean
    out["demand_mean_for_proxy"] = demand_mean
    out["delivered_volume_proxy_mw"] = delivered_volume_proxy(
        out["selected_profile_for_switch"], volume, generation_mean, demand_mean
    )
    out["seller_metric"] = seller_metric
    out["buyer_metric"] = buyer_metric
    out["total_metric"] = seller_metric + buyer_metric
    out["buyer_participation_slack_metric"] = buyer_slack

    # Decision signature uses rounded terms to avoid false mismatches from floating-point formatting.
    out["decision_signature"] = (
        out["ppa_type_normalized"].astype(str) + "|" +
        out["selected_profile_for_switch"].astype(str) + "|" +
        out["selected_strike_price_mwh"].round(6).astype(str) + "|" +
        out["selected_volume_mw"].round(6).astype(str)
    )

    return out, sources


def apply_filters(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if filter_risk_labels is not None:
        allowed = {str(x) for x in filter_risk_labels}
        out = out.loc[out["risk_label"].astype(str).isin(allowed)].copy()
    if filter_risk_modes is not None:
        allowed = {str(x) for x in filter_risk_modes}
        out = out.loc[out["risk_mode"].astype(str).isin(allowed)].copy()
    if filter_lambda_s_values is not None:
        out = out.loc[_numeric_filter(out["lambda_s"], filter_lambda_s_values)].copy()
    if filter_lambda_b_values is not None:
        out = out.loc[_numeric_filter(out["lambda_b"], filter_lambda_b_values)].copy()
    if filter_match_ids is not None:
        allowed = {int(x) for x in filter_match_ids}
        out = out.loc[out["match_id"].astype(int).isin(allowed)].copy()
    if filter_mutation_families is not None:
        allowed_raw = {str(x).lower() for x in filter_mutation_families}
        allowed_norm = {normalize_family_value(x) for x in filter_mutation_families}
        is_baseline = out["scenario_type"].astype(str).eq("baseline")
        keep = is_baseline | out["mutation_family"].astype(str).str.lower().isin(allowed_raw) | out["mutation_family_normalized"].isin(allowed_norm)
        out = out.loc[keep].copy()
    if filter_target_physical_shifts is not None:
        is_baseline = out["scenario_type"].astype(str).eq("baseline")
        keep = is_baseline | _numeric_filter(out["target_shift_signed"], filter_target_physical_shifts)
        out = out.loc[keep].copy()
    if filter_scenario_names is not None:
        allowed = {str(x) for x in filter_scenario_names}
        is_baseline = out["scenario_type"].astype(str).eq("baseline")
        out = out.loc[is_baseline | out["scenario_name"].astype(str).isin(allowed)].copy()
    return out.reset_index(drop=True)


def filter_to_success_rows(df: pd.DataFrame, status: pd.DataFrame) -> pd.DataFrame:
    if not require_success_in_run_status or status.empty:
        return df
    required = ["risk_label", "lambda_s", "lambda_b", "match_id", "scenario_name", "status"]
    missing = [c for c in required if c not in status.columns]
    if missing:
        print(f"Success filter skipped; run-status table is missing columns: {missing}")
        return df
    ok = status.loc[status["status"].astype(str).eq("success"), ["risk_label", "lambda_s", "lambda_b", "match_id", "scenario_name"]].copy()
    for col in ["lambda_s", "lambda_b"]:
        ok[col] = _num(ok[col]).round(9)
    out = df.copy()
    out["lambda_s_join"] = out["lambda_s"].round(9)
    out["lambda_b_join"] = out["lambda_b"].round(9)
    ok = ok.rename(columns={"lambda_s": "lambda_s_join", "lambda_b": "lambda_b_join"}).drop_duplicates()
    out = out.merge(ok, on=["risk_label", "lambda_s_join", "lambda_b_join", "match_id", "scenario_name"], how="inner")
    return out.drop(columns=["lambda_s_join", "lambda_b_join"], errors="ignore")


summary_std, column_sources = standardize_summary(summary_raw)
summary_std = filter_to_success_rows(summary_std, run_status_raw)
analysis_df = apply_filters(summary_std)

print("Column sources:")
for key, value in column_sources.items():
    print(f"  {key}: {value}")
print()
print("Rows after standardization and filters:", len(analysis_df))
print("Risk settings:", analysis_df["risk_label_display"].nunique())
print("Matches:", analysis_df["match_id"].nunique())
print("Scenarios:", analysis_df["scenario_name"].nunique())

In [ ]:
# ============================================================
# SECTION 6. BUILD PAIRED BASELINE-MUTATION CHANGE TABLE
# ============================================================

def build_paired_change_table(df: pd.DataFrame) -> pd.DataFrame:
    baseline = df.loc[df["scenario_type"].eq("baseline")].copy()
    mutation = df.loc[df["scenario_type"].eq("mutation")].copy()
    if baseline.empty:
        raise ValueError("No baseline rows were found. Baseline rows are required for paired mutation deltas.")
    if mutation.empty:
        raise ValueError("No mutation rows were found after filters.")

    base_cols = [
        "risk_setting_key", "match_id", "ppa_type_normalized", "profile_type_normalized", "selected_profile_for_switch",
        "active_contract", "decision_signature", "selected_strike_price_mwh", "selected_volume_mw",
        "delivered_volume_proxy_mw", "seller_metric", "buyer_metric", "total_metric",
        "buyer_participation_slack_metric", "generation_mean_for_proxy", "demand_mean_for_proxy",
    ]
    base_cols = [c for c in base_cols if c in baseline.columns]
    baseline_small = baseline[base_cols].drop_duplicates(["risk_setting_key", "match_id"], keep="first")
    baseline_small = baseline_small.rename(columns={c: f"baseline_{c}" for c in base_cols if c not in {"risk_setting_key", "match_id"}})

    paired = mutation.merge(baseline_small, on=["risk_setting_key", "match_id"], how="left", validate="many_to_one")
    paired["has_same_risk_baseline"] = paired["baseline_decision_signature"].notna()

    paired["contract_type_changed"] = paired["ppa_type_normalized"].astype(str) != paired["baseline_ppa_type_normalized"].astype(str)
    paired["profile_changed"] = paired["selected_profile_for_switch"].astype(str) != paired["baseline_selected_profile_for_switch"].astype(str)
    paired["full_decision_changed"] = paired["decision_signature"].astype(str) != paired["baseline_decision_signature"].astype(str)
    paired["ppa_formation_rate_current"] = paired["active_contract"].astype(float)
    paired["ppa_formation_rate_baseline"] = paired["baseline_active_contract"].astype(float)
    paired["ppa_formation_changed"] = paired["active_contract"].astype(str) != paired["baseline_active_contract"].astype(str)
    paired["fix_to_fix"] = paired["selected_profile_for_switch"].eq("Fix") & paired["baseline_selected_profile_for_switch"].eq("Fix")

    paired["delta_strike_price_mwh"] = paired["selected_strike_price_mwh"] - paired["baseline_selected_strike_price_mwh"]
    paired["delta_contract_volume_mw"] = paired["selected_volume_mw"] - paired["baseline_selected_volume_mw"]
    paired["delta_fixed_volume_mw"] = paired["delta_contract_volume_mw"].where(paired["fix_to_fix"])
    paired["delta_delivered_volume_proxy_mw"] = paired["delivered_volume_proxy_mw"] - paired["baseline_delivered_volume_proxy_mw"]
    paired["delta_seller_metric"] = paired["seller_metric"] - paired["baseline_seller_metric"]
    paired["delta_buyer_metric"] = paired["buyer_metric"] - paired["baseline_buyer_metric"]
    paired["delta_total_metric"] = paired["total_metric"] - paired["baseline_total_metric"]
    paired["delta_buyer_participation_slack"] = paired["buyer_participation_slack_metric"] - paired["baseline_buyer_participation_slack_metric"]

    return paired.sort_values(["risk_setting_order", "risk_label", "mutation_family_normalized", "target_shift_signed", "match_id"]).reset_index(drop=True)


paired_df = build_paired_change_table(analysis_df)

# Standardized all-row export.
standardized_export = analysis_df.copy()
if not include_baseline_rows_in_standardized_export:
    standardized_export = standardized_export.loc[standardized_export["scenario_type"].eq("mutation")].copy()

standardized_export.to_csv(TABLE_DIR / "RA_Mutation_Standardized_All_Rows.csv", index=False)
paired_df.to_csv(TABLE_DIR / "RA_Mutation_Paired_Baseline_Changes_Long.csv", index=False)

print("Paired mutation rows:", len(paired_df))
print("Rows with matched same-risk baseline:", int(paired_df["has_same_risk_baseline"].sum()))
print("Profile switch rate:", round(float(paired_df["profile_changed"].mean()), 4))
print("Contract-type switch rate:", round(float(paired_df["contract_type_changed"].mean()), 4))

In [ ]:
# ============================================================
# SECTION 7. SUMMARY TABLE BUILDERS
# ============================================================



def _dedupe_list(values: list[str]) -> list[str]:
    out = []
    for v in values:
        if v not in out:
            out.append(v)
    return out


def ordered_unique(values, preferred_order=None) -> list:
    vals = [v for v in pd.Series(values).dropna().astype(str).unique().tolist() if v != ""]
    if preferred_order is None:
        return sorted(vals)
    order = {v: i for i, v in enumerate(preferred_order)}
    return sorted(vals, key=lambda x: (order.get(x, len(order) + 1), x))


def summarize_run_status(status: pd.DataFrame) -> pd.DataFrame:
    if status.empty:
        return pd.DataFrame()
    out = status.copy()
    out["risk_mode"] = [risk_mode(s, b) for s, b in zip(_num(out.get("lambda_s", 0)).fillna(0), _num(out.get("lambda_b", 0)).fillna(0))]
    group_cols = ["risk_label", "lambda_s", "lambda_b", "risk_mode", "status"]
    counts = out.groupby(group_cols, dropna=False).size().reset_index(name="n_rows")
    totals = counts.groupby(["risk_label", "lambda_s", "lambda_b"], dropna=False)["n_rows"].transform("sum")
    counts["share"] = counts["n_rows"] / totals
    return counts.sort_values(["risk_label", "status"]).reset_index(drop=True)


def build_profile_share_table(df: pd.DataFrame) -> pd.DataFrame:
    group_cols = [
        "risk_setting_order", "risk_label", "risk_label_display", "lambda_s", "lambda_b", "risk_mode", "risk_intensity",
        "scenario_name", "scenario_type", "scenario_order" if "scenario_order" in df.columns else "scenario_name",
        "mutation_family", "mutation_family_normalized", "target_shift_signed", "target_shift_label",
        "selected_profile_for_switch",
    ]
    group_cols = _dedupe_list([c for c in group_cols if c in df.columns])
    counts = df.groupby(group_cols, dropna=False).size().reset_index(name="n_matches_profile")
    denom_cols = [c for c in group_cols if c != "selected_profile_for_switch"]
    counts["n_matches_scenario"] = counts.groupby(denom_cols, dropna=False)["n_matches_profile"].transform("sum")
    counts["profile_share"] = counts["n_matches_profile"] / counts["n_matches_scenario"]
    return counts.sort_values(["risk_setting_order", "mutation_family_normalized", "target_shift_signed", "selected_profile_for_switch"]).reset_index(drop=True)


def _metric_stats(series: pd.Series) -> dict:
    x = _num(series).replace([np.inf, -np.inf], np.nan).dropna()
    if x.empty:
        return {"n": 0, "mean": np.nan, "std": np.nan, "median": np.nan, "q10": np.nan, "q25": np.nan, "q75": np.nan, "q90": np.nan}
    return {
        "n": int(x.size),
        "mean": float(x.mean()),
        "std": float(x.std(ddof=1)) if x.size > 1 else 0.0,
        "median": float(x.median()),
        "q10": float(x.quantile(0.10)),
        "q25": float(x.quantile(0.25)),
        "q75": float(x.quantile(0.75)),
        "q90": float(x.quantile(0.90)),
    }


def build_case_summary(paired: pd.DataFrame) -> pd.DataFrame:
    group_cols = [
        "risk_setting_order", "risk_label", "risk_label_display", "lambda_s", "lambda_b", "risk_mode", "risk_intensity",
        "scenario_name", "scenario_order" if "scenario_order" in paired.columns else "scenario_name",
        "mutation_family", "mutation_family_normalized", "target_shift_signed", "target_shift_label",
    ]
    group_cols = _dedupe_list([c for c in group_cols if c in paired.columns])
    rows = []
    for keys, sub in paired.groupby(group_cols, dropna=False):
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        row["n_matches"] = int(len(sub))
        row["n_with_baseline"] = int(sub["has_same_risk_baseline"].sum()) if "has_same_risk_baseline" in sub.columns else int(len(sub))
        row["ppa_formation_rate"] = float(sub["active_contract"].mean()) if "active_contract" in sub.columns else np.nan
        row["baseline_ppa_formation_rate"] = float(sub["baseline_active_contract"].mean()) if "baseline_active_contract" in sub.columns else np.nan
        row["profile_switch_rate"] = float(sub["profile_changed"].mean()) if "profile_changed" in sub.columns else np.nan
        row["contract_type_switch_rate"] = float(sub["contract_type_changed"].mean()) if "contract_type_changed" in sub.columns else np.nan
        row["full_decision_switch_rate"] = float(sub["full_decision_changed"].mean()) if "full_decision_changed" in sub.columns else np.nan
        for profile in profile_order:
            row[f"share_{profile.replace(' ', '_')}"] = float(sub["selected_profile_for_switch"].eq(profile).mean())
            row[f"baseline_share_{profile.replace(' ', '_')}"] = float(sub["baseline_selected_profile_for_switch"].eq(profile).mean()) if "baseline_selected_profile_for_switch" in sub.columns else np.nan
        for metric in metrics_to_summarize:
            if metric in sub.columns:
                stats = _metric_stats(sub[metric])
                for stat, value in stats.items():
                    row[f"{metric}_{stat}"] = value
        rows.append(row)
    return pd.DataFrame(rows).sort_values(["risk_setting_order", "mutation_family_normalized", "target_shift_signed"]).reset_index(drop=True)


def build_metric_distribution_long(paired: pd.DataFrame) -> pd.DataFrame:
    group_cols = [
        "risk_setting_order", "risk_label", "risk_label_display", "lambda_s", "lambda_b", "risk_mode", "risk_intensity",
        "scenario_name", "mutation_family_normalized", "target_shift_signed", "target_shift_label",
    ]
    group_cols = _dedupe_list([c for c in group_cols if c in paired.columns])
    rows = []
    for metric in metrics_to_summarize:
        if metric not in paired.columns:
            continue
        for keys, sub in paired.groupby(group_cols, dropna=False):
            row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
            row["metric"] = metric
            row["metric_label"] = metric_display_labels.get(metric, metric)
            row.update(_metric_stats(sub[metric]))
            rows.append(row)
    return pd.DataFrame(rows)


def build_transition_table(paired: pd.DataFrame) -> pd.DataFrame:
    cols = [
        "risk_setting_order", "risk_label", "risk_label_display", "lambda_s", "lambda_b", "risk_mode", "risk_intensity",
        "scenario_name", "mutation_family_normalized", "target_shift_signed", "target_shift_label",
        "baseline_selected_profile_for_switch", "selected_profile_for_switch",
    ]
    cols = _dedupe_list([c for c in cols if c in paired.columns])
    out = paired.groupby(cols, dropna=False).size().reset_index(name="count")
    denom_cols = [c for c in cols if c != "selected_profile_for_switch"]
    out["baseline_profile_count"] = out.groupby(denom_cols, dropna=False)["count"].transform("sum")
    out["share_within_baseline_profile"] = out["count"] / out["baseline_profile_count"]
    return out.sort_values(["risk_setting_order", "mutation_family_normalized", "target_shift_signed"]).reset_index(drop=True)


def build_severe_target_table(case_summary: pd.DataFrame) -> pd.DataFrame:
    if case_summary.empty:
        return pd.DataFrame()
    rows = []
    group_cols = ["risk_label", "lambda_s", "lambda_b", "risk_mode", "mutation_family_normalized"]
    for _, sub in case_summary.groupby(group_cols, dropna=False):
        s = sub.copy()
        s["_abs_shift"] = _num(s["target_shift_signed"]).abs()
        max_abs = s["_abs_shift"].max()
        rows.append(s.loc[np.isclose(s["_abs_shift"], max_abs)].sort_values("target_shift_signed").tail(1).drop(columns="_abs_shift"))
    return pd.concat(rows, ignore_index=True, sort=False) if rows else pd.DataFrame()


run_status_summary = summarize_run_status(run_status_raw)
profile_share_tbl = build_profile_share_table(analysis_df)
case_summary_tbl = build_case_summary(paired_df)
metric_distribution_tbl = build_metric_distribution_long(paired_df)
transition_tbl = build_transition_table(paired_df)
severe_case_summary_tbl = build_severe_target_table(case_summary_tbl)

exports = {
    "RA_Mutation_Run_Status_Summary.csv": run_status_summary,
    "RA_Mutation_Profile_Share_ByRiskScenario.csv": profile_share_tbl,
    "RA_Mutation_Case_Summary_ByRiskScenario.csv": case_summary_tbl,
    "RA_Mutation_Metric_Distribution_Long.csv": metric_distribution_tbl,
    "RA_Mutation_Profile_Transition_Long.csv": transition_tbl,
    "RA_Mutation_Severe_Target_Case_Summary.csv": severe_case_summary_tbl,
}
for name, table in exports.items():
    table.to_csv(TABLE_DIR / name, index=False, float_format="%.8g")

print("Exported summary tables:")
for name, table in exports.items():
    print(f"  {name}: {len(table)} rows")

In [ ]:
# ============================================================
# SECTION 8. OPTIONAL RISK-NEUTRAL REFERENCE COMPARISON
# ============================================================

def resolve_risk_neutral_plot_data() -> Optional[Path]:
    override = _as_path_or_none(risk_neutral_plot_data_csv_override)
    if override is not None:
        return override if override.exists() else None
    candidates = [
        CODE_ROOT / risk_neutral_folder_name / risk_neutral_output_folder_name / risk_neutral_plot_data_filename,
        NOTEBOOK_CWD / risk_neutral_folder_name / risk_neutral_output_folder_name / risk_neutral_plot_data_filename,
        NOTEBOOK_CWD.parent / risk_neutral_folder_name / risk_neutral_output_folder_name / risk_neutral_plot_data_filename,
        RISK_AVERSE_DIR.parent / risk_neutral_folder_name / risk_neutral_output_folder_name / risk_neutral_plot_data_filename,
    ]
    for c in _dedupe_paths(candidates):
        if c.exists():
            return c
    return None


def build_risk_neutral_comparison(ra_df: pd.DataFrame) -> pd.DataFrame:
    rn_path = resolve_risk_neutral_plot_data()
    if not include_risk_neutral_reference or rn_path is None:
        if include_risk_neutral_reference:
            print("Risk-neutral reference not found; skipping risk-neutral comparison.")
        return pd.DataFrame()
    rn_raw = pd.read_csv(rn_path)
    rn_raw["risk_label"] = "risk_neutral_reference"
    rn_raw["lambda_s"] = 0.0
    rn_raw["lambda_b"] = 0.0
    rn_std, _ = standardize_summary(rn_raw)
    rn_std["rn_join_shift"] = rn_std["target_shift_signed"].round(9)
    rn_small_cols = [
        "match_id", "scenario_name", "mutation_family_normalized", "rn_join_shift",
        "ppa_type_normalized", "selected_profile_for_switch", "decision_signature",
        "selected_strike_price_mwh", "selected_volume_mw", "delivered_volume_proxy_mw",
        "seller_metric", "buyer_metric", "total_metric", "buyer_participation_slack_metric",
    ]
    rn_small_cols = [c for c in rn_small_cols if c in rn_std.columns]
    rn_small = rn_std[rn_small_cols].rename(columns={c: f"rn_{c}" for c in rn_small_cols if c not in {"match_id", "scenario_name", "mutation_family_normalized", "rn_join_shift"}})

    ra = ra_df.copy()
    ra["rn_join_shift"] = ra["target_shift_signed"].round(9)
    comp = ra.merge(
        rn_small,
        on=["match_id", "scenario_name", "mutation_family_normalized", "rn_join_shift"],
        how="left",
    )
    comp["has_risk_neutral_reference"] = comp["rn_decision_signature"].notna()
    comp["different_profile_than_risk_neutral"] = comp["selected_profile_for_switch"].astype(str) != comp["rn_selected_profile_for_switch"].astype(str)
    comp["different_full_decision_than_risk_neutral"] = comp["decision_signature"].astype(str) != comp["rn_decision_signature"].astype(str)
    comp["ra_minus_rn_seller_metric"] = comp["seller_metric"] - comp["rn_seller_metric"]
    comp["ra_minus_rn_buyer_metric"] = comp["buyer_metric"] - comp["rn_buyer_metric"]
    comp["ra_minus_rn_total_metric"] = comp["total_metric"] - comp["rn_total_metric"]
    comp["ra_minus_rn_strike_price_mwh"] = comp["selected_strike_price_mwh"] - comp["rn_selected_strike_price_mwh"]
    comp["ra_minus_rn_volume_mw"] = comp["selected_volume_mw"] - comp["rn_selected_volume_mw"]
    print("Loaded risk-neutral reference:", rn_path)
    print("Risk-neutral matched RA rows:", int(comp["has_risk_neutral_reference"].sum()), "/", len(comp))
    return comp


def summarize_risk_neutral_comparison(comp: pd.DataFrame) -> pd.DataFrame:
    if comp.empty:
        return pd.DataFrame()
    group_cols = [
        "risk_setting_order", "risk_label", "risk_label_display", "lambda_s", "lambda_b", "risk_mode", "risk_intensity",
        "scenario_name", "mutation_family_normalized", "target_shift_signed", "target_shift_label",
    ]
    group_cols = [c for c in group_cols if c in comp.columns]
    rows = []
    for keys, sub in comp.groupby(group_cols, dropna=False):
        sub = sub.loc[sub["has_risk_neutral_reference"]].copy()
        row = dict(zip(group_cols, keys if isinstance(keys, tuple) else (keys,)))
        row["n_matched_reference"] = int(len(sub))
        if len(sub) == 0:
            rows.append(row)
            continue
        row["different_profile_than_risk_neutral_rate"] = float(sub["different_profile_than_risk_neutral"].mean())
        row["different_full_decision_than_risk_neutral_rate"] = float(sub["different_full_decision_than_risk_neutral"].mean())
        for metric in ["ra_minus_rn_seller_metric", "ra_minus_rn_buyer_metric", "ra_minus_rn_total_metric", "ra_minus_rn_strike_price_mwh", "ra_minus_rn_volume_mw"]:
            stats = _metric_stats(sub[metric])
            for stat, value in stats.items():
                row[f"{metric}_{stat}"] = value
        rows.append(row)
    return pd.DataFrame(rows)


risk_neutral_comparison_df = build_risk_neutral_comparison(analysis_df)
risk_neutral_summary_tbl = summarize_risk_neutral_comparison(risk_neutral_comparison_df)
if not risk_neutral_comparison_df.empty:
    risk_neutral_comparison_df.to_csv(TABLE_DIR / "RA_vs_RiskNeutral_Row_Comparison.csv", index=False)
    risk_neutral_summary_tbl.to_csv(TABLE_DIR / "RA_vs_RiskNeutral_Summary_ByRiskScenario.csv", index=False, float_format="%.8g")

In [ ]:
# ============================================================
# SECTION 9. FIGURE HELPERS
# ============================================================

def _close_or_show(fig):
    if display_figures_in_notebook:
        display(fig)
    plt.close(fig)


def _save_figure(fig, stem: str):
    if not make_figures:
        return
    saved = []
    if save_png:
        path = FIGURE_DIR / f"{stem}.png"
        fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
        saved.append(path)
    if save_pdf:
        path = FIGURE_DIR / f"{stem}.pdf"
        fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
        saved.append(path)
    if save_svg:
        path = FIGURE_DIR / f"{stem}.svg"
        fig.savefig(path, dpi=figure_dpi, bbox_inches="tight")
        saved.append(path)
    if saved:
        print("Saved", stem, "->", ", ".join(p.name for p in saved))


def _risk_sort_key(label: str, lambda_s: float, lambda_b: float, order=np.nan):
    mode = risk_mode(lambda_s, lambda_b)
    return (risk_mode_order.index(mode) if mode in risk_mode_order else 999, float(max(abs(lambda_s), abs(lambda_b))), order if pd.notna(order) else 999, label)


def ordered_risk_rows(df: pd.DataFrame) -> pd.DataFrame:
    cols = ["risk_setting_order", "risk_label", "risk_label_display", "lambda_s", "lambda_b", "risk_mode", "risk_intensity"]
    cols = [c for c in cols if c in df.columns]
    rows = df[cols].drop_duplicates().copy()
    rows["_sort"] = rows.apply(lambda r: _risk_sort_key(str(r.get("risk_label", "")), r.get("lambda_s", 0), r.get("lambda_b", 0), r.get("risk_setting_order", np.nan)), axis=1)
    return rows.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)


def ordered_families(df: pd.DataFrame) -> list[str]:
    return ordered_unique(df.get("mutation_family_normalized", pd.Series(dtype=str)), family_order)


def scenario_column_label(family: str, shift_label: str) -> str:
    return f"{family}\n{shift_label}"


def pivot_scenario_matrix(tbl: pd.DataFrame, value_col: str, index_col: str = "risk_label_display") -> pd.DataFrame:
    t = tbl.copy()
    t["_family_order"] = t["mutation_family_normalized"].map(lambda x: family_order.index(x) if x in family_order else 999)
    t["_shift_sort"] = _num(t["target_shift_signed"])
    t["scenario_col"] = [scenario_column_label(f, s) for f, s in zip(t["mutation_family_normalized"], t["target_shift_label"])]
    order = t.sort_values(["_family_order", "_shift_sort"])["scenario_col"].drop_duplicates().tolist()
    mat = t.pivot_table(index=index_col, columns="scenario_col", values=value_col, aggfunc="mean")
    mat = mat.reindex(columns=[c for c in order if c in mat.columns])
    risk_order_df = ordered_risk_rows(t)
    idx_order = risk_order_df[index_col].tolist() if index_col in risk_order_df.columns else mat.index.tolist()
    mat = mat.reindex([x for x in idx_order if x in mat.index])
    return mat


def draw_heatmap(mat: pd.DataFrame, colorbar_label: str, stem: str, value_format: str = ".2g"):
    if mat.empty:
        print(f"Skipped {stem}: empty matrix")
        return
    width = max(default_fig_width, min(24.0, 0.85 * len(mat.columns) + 3.0))
    height = max(minimum_fig_height, 0.45 * len(mat.index) + 2.5)
    fig, ax = plt.subplots(figsize=(width, height), dpi=figure_dpi)
    im = ax.imshow(mat.to_numpy(dtype=float), aspect="auto")
    ax.set_xticks(np.arange(len(mat.columns)))
    ax.set_xticklabels(mat.columns, rotation=x_tick_rotation, ha="right", fontsize=tick_label_fontsize)
    ax.set_yticks(np.arange(len(mat.index)))
    ax.set_yticklabels(mat.index, fontsize=tick_label_fontsize)
    cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
    cbar.set_label(colorbar_label, fontsize=axis_label_fontsize)
    cbar.ax.tick_params(labelsize=tick_label_fontsize)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            val = mat.iloc[i, j]
            if pd.notna(val):
                try:
                    label = format(float(val), value_format)
                except Exception:
                    label = str(val)
                ax.text(j, i, label, ha="center", va="center", fontsize=heatmap_annotation_fontsize)
    fig.tight_layout()
    _save_figure(fig, stem)
    _close_or_show(fig)


def _safe_stem(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(text)).strip("_")


def wrap_panel_title(title: str, width: int = 28) -> str:
    """Wrap a panel title at word boundaries to fit its subplot."""
    return "\n".join(textwrap.wrap(str(title), width=width, break_long_words=False, break_on_hyphens=False))

In [ ]:
# ============================================================
# SECTION 10. FIGURE BUILDERS
# ============================================================

def plot_run_status(status_summary: pd.DataFrame):
    if status_summary.empty:
        print("Skipped run-status figure: no run-status table loaded.")
        return
    mat = status_summary.pivot_table(index="risk_label", columns="status", values="n_rows", aggfunc="sum", fill_value=0)
    order = ordered_risk_rows(status_summary)["risk_label"].tolist()
    mat = mat.reindex([x for x in order if x in mat.index])
    fig, ax = plt.subplots(figsize=(max(default_fig_width, 0.8 * len(mat.index)), 5.2), dpi=figure_dpi)
    mat.plot(kind="bar", stacked=True, ax=ax)
    ax.set_xlabel("Risk setting", fontsize=axis_label_fontsize)
    ax.set_ylabel("Run-status rows", fontsize=axis_label_fontsize)
    ax.tick_params(axis="x", labelrotation=x_tick_rotation, labelsize=tick_label_fontsize)
    ax.tick_params(axis="y", labelsize=tick_label_fontsize)
    ax.legend(fontsize=legend_fontsize)
    fig.tight_layout()
    _save_figure(fig, "fig_run_status_by_risk")
    _close_or_show(fig)


def plot_baseline_profile_by_risk(df: pd.DataFrame):
    base = df.loc[df["scenario_type"].eq("baseline")].copy()
    if base.empty:
        print("Skipped baseline profile figure: no baseline rows.")
        return
    prof = build_profile_share_table(base)
    modes = ordered_unique(prof["risk_mode"], risk_mode_order)
    ncols = 1
    nrows = len(modes)
    fig, axes = plt.subplots(nrows, ncols, figsize=(default_fig_width, max(minimum_fig_height, default_row_height * nrows)), dpi=figure_dpi, squeeze=False)
    for r, mode in enumerate(modes):
        ax = axes[r, 0]
        sub = prof.loc[prof["risk_mode"].eq(mode)].copy()
        for profile in profile_order:
            s = sub.loc[sub["selected_profile_for_switch"].eq(profile)].sort_values("risk_intensity")
            if s.empty:
                continue
            ax.plot(s["risk_intensity"], s["profile_share"], marker="o", label=profile)
        ax.set_xlabel("Risk intensity max(λs, λb)", fontsize=axis_label_fontsize)
        ax.set_ylabel("Baseline selected-profile share", fontsize=axis_label_fontsize)
        ax.yaxis.set_major_formatter(PercentFormatter(1.0))
        ax.tick_params(labelsize=tick_label_fontsize)
        ax.grid(True, alpha=0.25)
        ax.legend(fontsize=legend_fontsize, frameon=True)
    fig.tight_layout()
    _save_figure(fig, "fig_baseline_profile_share_by_risk")
    _close_or_show(fig)


def plot_profile_share_heatmaps(profile_share: pd.DataFrame):
    mut = profile_share.loc[profile_share["scenario_type"].astype(str).eq("mutation")].copy()
    if mut.empty:
        print("Skipped profile-share heatmaps: no mutation rows.")
        return
    for profile in profile_share_heatmap_profiles:
        sub = mut.loc[mut["selected_profile_for_switch"].eq(profile)].copy()
        if sub.empty:
            print(f"Skipped profile-share heatmap for {profile}: no rows.")
            continue
        mat = pivot_scenario_matrix(sub, "profile_share", index_col="risk_label_display")
        if mat.shape[1] > profile_share_heatmap_max_columns:
            mat = mat.iloc[:, :profile_share_heatmap_max_columns]
        draw_heatmap(
            mat,
            colorbar_label="share",
            stem=f"fig_profile_share_heatmap_{_safe_stem(profile)}",
            value_format=heatmap_annotation_format,
        )


def plot_switch_rate_heatmap(case_summary: pd.DataFrame):
    mut = case_summary.copy()
    if mut.empty or switch_heatmap_value not in mut.columns:
        print("Skipped switch-rate heatmap: missing case summary or value column.")
        return
    mat = pivot_scenario_matrix(mut, switch_heatmap_value, index_col="risk_label_display")
    draw_heatmap(
        mat,
        colorbar_label="rate",
        stem=f"fig_heatmap_{switch_heatmap_value}",
        value_format=heatmap_annotation_format,
    )


def plot_metric_interval_figures(metric_dist: pd.DataFrame):
    if metric_dist.empty:
        print("Skipped metric interval figures: empty metric distribution table.")
        return
    metrics = [m for m in metric_interval_figure_metrics if m in metric_dist["metric"].unique()]
    if metric_interval_risk_modes is None:
        modes = ordered_unique(metric_dist["risk_mode"], risk_mode_order)
    elif isinstance(metric_interval_risk_modes, str):
        modes = [metric_interval_risk_modes]
    else:
        modes = list(metric_interval_risk_modes)
    for metric in metrics:
        for mode in modes:
            sub = metric_dist.loc[(metric_dist["metric"].eq(metric)) & (metric_dist["risk_mode"].eq(mode))].copy()
            if sub.empty:
                continue
            families = ordered_families(sub)
            ncols = len(families)
            panel_width = metric_interval_joint_panel_width if mode == "Joint risk-averse" else metric_interval_panel_width
            fig, axes = plt.subplots(1, ncols, figsize=(max(default_fig_width, panel_width * ncols), metric_interval_figure_height), dpi=figure_dpi, squeeze=False)
            for c, fam in enumerate(families):
                ax = axes[0, c]
                fs = sub.loc[sub["mutation_family_normalized"].eq(fam)].copy()
                for risk_label, rs in fs.groupby("risk_label_display", dropna=False):
                    rs = rs.sort_values("target_shift_signed")
                    y = rs[metric_interval_plot_stat]
                    raw_label = rs["risk_label"].iloc[0] if "risk_label" in rs.columns else risk_label
                    legend_label = risk_legend_label(raw_label, rs["lambda_s"].iloc[0], rs["lambda_b"].iloc[0])
                    ax.plot(rs["target_shift_signed"], y, marker="o", label=legend_label)
                    if metric_interval_error_band in {"iqr", "p10_p90"}:
                        lo, hi = ("q25", "q75") if metric_interval_error_band == "iqr" else ("q10", "q90")
                        if lo in rs.columns and hi in rs.columns:
                            ax.fill_between(rs["target_shift_signed"].to_numpy(dtype=float), rs[lo].to_numpy(dtype=float), rs[hi].to_numpy(dtype=float), alpha=0.15)
                ax.axhline(0, linewidth=0.8)
                panel_title = wrap_panel_title(family_panel_title_labels.get(fam, fam))
                ax.set_title(panel_title, fontsize=metric_interval_panel_title_fontsize, loc="center", y=1.22, pad=0, verticalalignment="top")
                ax.set_xlabel("Correlation shift", fontsize=metric_interval_axis_label_fontsize, labelpad=5)
                if c == 0:
                    ax.set_ylabel(metric_display_labels.get(metric, metric), fontsize=metric_interval_axis_label_fontsize, labelpad=5)
                ax.tick_params(labelsize=metric_interval_tick_label_fontsize)
                ax.yaxis.get_offset_text().set_fontsize(metric_interval_tick_label_fontsize)
                ax.grid(True, alpha=0.25)
            handles, labels = axes[0, 0].get_legend_handles_labels()
            fig.legend(handles, labels, fontsize=metric_interval_legend_fontsize, frameon=True, loc="lower center", bbox_to_anchor=(0.5, 0.015), ncol=min(2, len(labels)), columnspacing=1.2, handlelength=2.0)
            fig.subplots_adjust(left=0.075, right=0.99, bottom=0.24, top=0.86, wspace=metric_interval_wspace)
            _save_figure(fig, f"fig_metric_interval_{_safe_stem(mode)}_{_safe_stem(metric)}")
            _close_or_show(fig)


def plot_risk_response_figures(case_summary: pd.DataFrame):
    severe = build_severe_target_table(case_summary)
    if severe.empty:
        print("Skipped risk-response figures: severe-target table is empty.")
        return
    for metric in risk_response_metrics:
        if metric not in severe.columns:
            continue
        modes = risk_response_risk_modes or ordered_unique(severe["risk_mode"], risk_mode_order)
        fig, axes = plt.subplots(len(modes), 1, figsize=(default_fig_width, max(minimum_fig_height, default_row_height * len(modes))), dpi=figure_dpi, squeeze=False)
        for r, mode in enumerate(modes):
            ax = axes[r, 0]
            sub = severe.loc[severe["risk_mode"].eq(mode)].copy()
            if sub.empty:
                ax.axis("off")
                continue
            for fam, fs in sub.groupby("mutation_family_normalized", dropna=False):
                fs = fs.sort_values("risk_intensity")
                ax.plot(fs["risk_intensity"], fs[metric], marker="o", label=str(fam))
            ax.set_xlabel("Risk intensity max(λs, λb)", fontsize=axis_label_fontsize)
            ax.set_ylabel(metric.replace("_", " "), fontsize=axis_label_fontsize)
            if "rate" in metric or "share" in metric:
                ax.yaxis.set_major_formatter(PercentFormatter(1.0))
            ax.grid(True, alpha=0.25)
            ax.tick_params(labelsize=tick_label_fontsize)
            ax.legend(fontsize=legend_fontsize, frameon=True, bbox_to_anchor=(1.02, 1.0), loc="upper left")
        fig.tight_layout()
        _save_figure(fig, f"fig_risk_response_{_safe_stem(metric)}")
        _close_or_show(fig)


def plot_risk_neutral_difference_figures(rn_summary: pd.DataFrame):
    if rn_summary.empty:
        return
    metric = "different_profile_than_risk_neutral_rate"
    if metric in rn_summary.columns:
        mat = pivot_scenario_matrix(rn_summary, metric, index_col="risk_label_display")
        draw_heatmap(
            mat,
            colorbar_label="share different",
            stem="fig_ra_vs_risk_neutral_profile_difference_heatmap",
            value_format=heatmap_annotation_format,
        )
    metric2 = "ra_minus_rn_total_metric_median"
    if metric2 in rn_summary.columns:
        mat = pivot_scenario_matrix(rn_summary, metric2, index_col="risk_label_display")
        draw_heatmap(
            mat,
            colorbar_label="RA − RN",
            stem="fig_ra_vs_risk_neutral_total_metric_heatmap",
            value_format=".2g",
        )

In [ ]:
# ============================================================
# SECTION 11. RUN FIGURES
# ============================================================
if make_figures:
    if make_run_status_figure:
        plot_run_status(run_status_summary)
    if make_baseline_profile_by_risk_figure:
        plot_baseline_profile_by_risk(analysis_df)
    if make_profile_share_heatmaps:
        plot_profile_share_heatmaps(profile_share_tbl)
    if make_switch_rate_heatmap:
        plot_switch_rate_heatmap(case_summary_tbl)
    if make_metric_delta_interval_figures:
        plot_metric_interval_figures(metric_distribution_tbl)
    if make_risk_response_figures:
        plot_risk_response_figures(case_summary_tbl)
    if make_risk_neutral_difference_figures:
        plot_risk_neutral_difference_figures(risk_neutral_summary_tbl)

print("Analysis complete.")
print(f"Tables : {TABLE_DIR}")
print(f"Figures: {FIGURE_DIR}")

print("\nMain table files:")
for p in sorted(TABLE_DIR.glob("*.csv")):
    print("  -", p.name)

print("\nMain figure files:")
for p in sorted(FIGURE_DIR.glob("fig_*")):
    print("  -", p.name)

In [ ]:
# ============================================================
# SECTION 12. PREVIEW TABLES AND BASIC INTERPRETATION AIDS
# ============================================================
if show_preview_tables:
    print("Risk-output inventory")
    display(risk_inventory.head(20))

    if not run_status_summary.empty:
        print("Run-status summary")
        display(run_status_summary.head(40))

    print("Case summary by risk/scenario")
    preview_cols = [
        "risk_label", "lambda_s", "lambda_b", "risk_mode", "mutation_family_normalized", "target_shift_label",
        "n_matches", "profile_switch_rate", "contract_type_switch_rate", "ppa_formation_rate",
        "delta_seller_metric_median", "delta_buyer_metric_median", "delta_buyer_participation_slack_median",
    ]
    preview_cols = [c for c in preview_cols if c in case_summary_tbl.columns]
    display(case_summary_tbl[preview_cols].head(40))

    print("Severe-target risk response summary")
    severe_cols = [
        "risk_label", "lambda_s", "lambda_b", "risk_mode", "mutation_family_normalized", "target_shift_label",
        "n_matches", "profile_switch_rate", "ppa_formation_rate",
        "delta_seller_metric_median", "delta_buyer_metric_median",
    ]
    severe_cols = [c for c in severe_cols if c in severe_case_summary_tbl.columns]
    display(severe_case_summary_tbl[severe_cols].head(40))

    if not risk_neutral_summary_tbl.empty:
        print("Risk-averse versus risk-neutral reference summary")
        rn_cols = [
            "risk_label", "risk_mode", "mutation_family_normalized", "target_shift_label", "n_matched_reference",
            "different_profile_than_risk_neutral_rate", "different_full_decision_than_risk_neutral_rate",
            "ra_minus_rn_total_metric_median",
        ]
        rn_cols = [c for c in rn_cols if c in risk_neutral_summary_tbl.columns]
        display(risk_neutral_summary_tbl[rn_cols].head(40))